In [2]:
!pip install webvtt-py

Importing Required Libraries

In [3]:
import pandas as pd
import re
import webvtt
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

Downloading NLTK Resources

In [4]:
import nltk

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

Reading and Previewing Comments Text File

In [5]:
with open('/kaggle/input/comments-and-caption/Comments.txt', 'r', encoding='utf-8') as f:
    for i in range(20):
        print(f.readline())

@milojadez

14 hours ago

There are people out there that actually are able to read minds and solve murders unexplainably, this guy just found his way to gain from it.

Reply





@JasonWestaway

22 hours ago

Wow, that was insane!

1

Reply





@THEHONESTREPLAY

1 day ago

That's all are fake,  Edited,  i challenged,

Reply





@serenaLei06



Reading and Previewing Caption VTT File

In [6]:
with open('/kaggle/input/comments-and-caption/Caption.vtt', 'r', encoding='utf-8') as f:
    for i in range(20):
        print(f.readline())

WEBVTT

Kind: captions

Language: en



00:00:04.240 --> 00:00:06.470 align:start position:0%

 

I<00:00:04.560><c> am</c><00:00:04.720><c> build</c><00:00:05.040><c> as</c><00:00:05.279><c> the</c><00:00:05.440><c> world's</c><00:00:05.759><c> greatest</c><00:00:06.160><c> mind</c>



00:00:06.470 --> 00:00:06.480 align:start position:0%

I am build as the world's greatest mind

 



00:00:06.480 --> 00:00:09.190 align:start position:0%

I am build as the world's greatest mind

readader.<00:00:07.279><c> But</c><00:00:07.520><c> guess</c><00:00:07.759><c> what?</c><00:00:08.639><c> I</c><00:00:08.880><c> can't</c><00:00:09.040><c> read</c>



00:00:09.190 --> 00:00:09.200 align:start position:0%

readader. But guess what? I can't read

 





Defining Function to Structure Comments from Text File

In [8]:
def structure_comments_from_txt(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    comments_data = []
    time_regex = re.compile(r'.*(second|minute|hour|day|week|month|year)s?\s+ago.*', re.IGNORECASE)

    i = 0
    while i < len(lines):
        line = lines[i].strip()

        if i+1 < len(lines) and time_regex.match(lines[i+1].strip()):
            username = line
            timestamp = lines[i+1].strip()
            comment_text = []

            i += 2
            while i < len(lines) and not (
                i+1 < len(lines) and time_regex.match(lines[i+1].strip())
            ):
                text = lines[i].strip()
                if text and text.lower() not in ['reply', '...more']:
                    comment_text.append(text)
                i += 1

            if comment_text:
                comments_data.append({
                    'username': username,
                    'timestamp_text': timestamp,
                    'comment_text': ' '.join(comment_text)
                })
        else:
            i += 1

    return pd.DataFrame(comments_data)


Comments Load

In [10]:
comments_df = structure_comments_from_txt('//kaggle/input/comments-and-caption/Comments.txt')
comments_df.head()

,username,timestamp_text,comment_text
0,@milojadez,14 hours ago,There are people out there that actually are a...
1,@JasonWestaway,22 hours ago,"Wow, that was insane! 1"
2,@THEHONESTREPLAY,1 day ago,"That's all are fake, Edited, i challenged,"
3,@serenaLei06,1 day ago,"Wait, how did he do that?"
4,@ainomugishanancy206,1 day ago,"If i know how people think,i know what they th..."


Defining Function to Structure Captions from VTT File

In [12]:
def structure_captions_from_vtt(filepath):
    captions = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if '-->' not in line and line and not line.isdigit() and 'WEBVTT' not in line:
                captions.append(line)

    full_text = ' '.join(captions)
    sentences = re.split(r'(?<=[.!?]) +', full_text)
    return pd.DataFrame(sentences, columns=['caption_sentence'])

Caption Load

In [13]:
captions_df = structure_captions_from_vtt('/kaggle/input/comments-and-caption/Caption.vtt')
captions_df.head()


,caption_sentence
0,Kind: captions Language: en I<00:00:04.560><c>...
1,But guess what?
2,I can't read readader.
3,But guess what?
4,I can't read minds.<00:00:10.480><c> What</c><...


Defining a Function to Remove VTT Formatting Tags

In [16]:
def remove_vtt_tags(text):
    # remove common VTT tags like <c>, </c>, <i>, </i>, <v Speaker>
    text = re.sub(r'<.*?>', '', text)
    return text

Normalize Text

In [20]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)          # remove brackets like [laugh]
    text = remove_vtt_tags(text)                 # remove VTT tags
    text = re.sub(r'[^a-z\s]', '', text)         # keep only letters and spaces
    text = text.strip()
    return text

Tokenization

In [21]:
def tokenize_text(text):
    return word_tokenize(text)

Stopword Removal

In [22]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words and len(word) > 2]

Lemmatization & Stemming

In [24]:
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

def stem_tokens(tokens):
    return [stemmer.stem(word) for word in tokens]

Full Cleaning Pipeline Function

In [26]:
def clean_text_pipeline(text, use_lemmatization=True):
    normalized = normalize_text(text)
    tokens = tokenize_text(normalized)
    filtered = remove_stopwords(tokens)

    if use_lemmatization:
        return lemmatize_tokens(filtered)
    else:
        return stem_tokens(filtered)

Apply on Comments DataFrame

In [27]:
comments_df['cleaned_tokens'] = comments_df['comment_text'].apply(
    lambda x: clean_text_pipeline(x, use_lemmatization=True)
)

comments_df.head()

,username,timestamp_text,comment_text,cleaned_tokens
0,@milojadez,14 hours ago,There are people out there that actually are a...,"[people, actually, able, read, mind, solve, mu..."
1,@JasonWestaway,22 hours ago,"Wow, that was insane! 1","[wow, insane]"
2,@THEHONESTREPLAY,1 day ago,"That's all are fake, Edited, i challenged,","[thats, fake, edited, challenged]"
3,@serenaLei06,1 day ago,"Wait, how did he do that?",[wait]
4,@ainomugishanancy206,1 day ago,"If i know how people think,i know what they th...","[know, people, thinki, know, think]"


Apply on Captions DataFrame

In [28]:
captions_df['cleaned_tokens'] = captions_df['caption_sentence'].apply(
    lambda x: clean_text_pipeline(x)
)

captions_df[['caption_sentence', 'cleaned_tokens']].head()

,caption_sentence,cleaned_tokens
0,Kind: captions Language: en I<00:00:04.560><c>...,"[kind, caption, language, build, world, greate..."
1,But guess what?,[guess]
2,I can't read readader.,"[cant, read, readader]"
3,But guess what?,[guess]
4,I can't read minds.<00:00:10.480><c> What</c><...,"[cant, read, mind, read, people, mind]"


Printing for verification:

In [29]:
# Check the first few rows of cleaned comments and captions
print("Comments cleaned tokens:")
print(comments_df.head())

print("\nCaptions cleaned tokens:")
print(captions_df.head())

Comments cleaned tokens:
               username timestamp_text  \
0            @milojadez   14 hours ago   
1        @JasonWestaway   22 hours ago   
2      @THEHONESTREPLAY      1 day ago   
3          @serenaLei06      1 day ago   
4  @ainomugishanancy206      1 day ago   

                                        comment_text  \
0  There are people out there that actually are a...   
1                            Wow, that was insane! 1   
2       That's all are fake,  Edited,  i challenged,   
3                          Wait, how did he do that?   
4  If i know how people think,i know what they th...   

                                      cleaned_tokens  
0  [people, actually, able, read, mind, solve, mu...  
1                                      [wow, insane]  
2                  [thats, fake, edited, challenged]  
3                                             [wait]  
4                [know, people, thinki, know, think]  

Captions cleaned tokens:
                             

Stemming vs Lemmatization

In [30]:
sample_sentence = "studies studying cries cry"
tokens = tokenize_text(sample_sentence)

print("Stemmed:", stem_tokens(tokens))
print("Lemmatized:", lemmatize_tokens(tokens))


Stemmed: ['studi', 'studi', 'cri', 'cri']
Lemmatized: ['study', 'studying', 'cry', 'cry']


Save cleaned DataFrames to CSV

In [31]:
import os

folder_path = '/kaggle/working/cleaned_data'

os.makedirs(folder_path, exist_ok=True)

print("Folder exists:", os.path.exists(folder_path))

Folder exists: True


In [32]:
comments_df.to_csv(f'{folder_path}/cleaned_comments.csv', index=False)
captions_df.to_csv(f'{folder_path}/cleaned_captions.csv', index=False)
print("Files in cleaned_data folder:", os.listdir(folder_path))

Files in cleaned_data folder: ['cleaned_comments.csv', 'cleaned_captions.csv']
